# 08 — Custom DL Baselines: RNA-only vs Protein-only (SimpleNeuralNetwork, DrugGNN)

Same pattern as notebook 07: independent of drevalpy, loads `response_pairs.parquet`
+ `data/splits/*.json` from notebook 04 directly, RNA-only and protein-only run
and reported separately, fusion deferred.

**Scope note:** `DIPK` is deliberately **not** included here — its drevalpy
implementation is ~1,100 lines across 5 files, includes a gene-expression
autoencoder *pretraining* stage, a custom attention mechanism, and BIONIC network
features that require an external feature pipeline this project doesn't have.
That's enough extra scope to warrant its own notebook if you want it — flag me.

**Architectures matched to drevalpy's own implementations** (verified from the
installed 1.5.1 source):
- [`SimpleNeuralNetwork/utils.py`](https://github.com/daisybio/drevalpy/blob/development/drevalpy/models/SimpleNeuralNetwork/utils.py)
  (`FeedForwardNetwork`)
- [`DrugGNN/drug_gnn.py`](https://github.com/daisybio/drevalpy/blob/development/drevalpy/models/DrugGNN/drug_gnn.py)
  (`DrugGraphNet`)

**Deviations from upstream, flagged explicitly:**
- Implemented with a plain PyTorch training loop instead of `pytorch_lightning` —
  same architecture and hyperparameters, just self-contained (no Lightning
  Trainer/callback machinery).
- `SimpleNeuralNetwork`'s drug view in drevalpy is `[fingerprints,
  drug_chemberta_embeddings]` — only fingerprints here (no ChemBERTa embedding
  pipeline in this project), matching notebook 07's convention.
- After hyperparameter selection, the early-stopped model from the *winning*
  trial is used directly for test prediction — **not** refit on train+val like
  the sklearn baselines in notebook 07, since retraining without a held-out set
  for early stopping risks overfitting on these architectures.

**Hyperparameter search and validation split logic match notebook 07 exactly**
(group-aware for LCO/LDO/LTO, plain random for LPO).

## Environment setup (Colab or local)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q torch-geometric rdkit
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## GPU check

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU detected — these models will be slow on CPU. "
          "Runtime > Change runtime type > GPU, then re-run this cell.")

## Imports and config

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch_geometric.data import Batch, Data
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

In [ ]:
DATA_DIR = BASE_PATH / "data" / "GDSC2"
PROCESSED_DIR = BASE_PATH / "data" / "processed"
SPLITS_DIR = BASE_PATH / "data" / "splits"
RESULTS_DIR = BASE_PATH / "results" / "custom_dl_baselines"

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

SPLIT_TYPES = ["lpo", "lco", "ldo", "lto"]
ARMS = ["rna", "protein"]

VALIDATION_RATIO = 0.1
TOP_K_FEATURES = 1000
RANDOM_STATE = 42

# Quick smoke test before committing to the full grid: limits to LCO, fold 0 only.
QUICK_TEST = True

## Load response pairs + splits (from notebook 04)

In [ ]:
pairs = pd.read_parquet(PROCESSED_DIR / "response_pairs.parquet")

# lpo/lco/ldo/lto.json: list of {"train": [...], "test": [...]} per fold, same
# format as before. As of notebook 04's redesign, "train" is now IDENTICAL
# across all four split types for a given fold index -- one shared train set,
# four different test sets (new cell lines / new drugs / new tissues / new
# pairs). Loading code below needs no change for that; flagging so it's not
# a surprise when e.g. splits["lco"][0]["train"] == splits["ldo"][0]["train"].
splits = {}
for split_type in SPLIT_TYPES:
    with open(SPLITS_DIR / f"{split_type}.json") as f:
        splits[split_type] = json.load(f)

# holdout_groups.json: which cell lines / drugs / tissues were actually held
# out per fold (entity IDs, not row indices) -- needed later to interpret
# results (e.g. which tissues drove a fold's LTO test set, whether the same
# drugs/cell lines are consistently hard across folds). Not used by the
# training/eval loops below, just loaded here so it's available alongside
# everything else.
with open(SPLITS_DIR / "holdout_groups.json") as f:
    holdout_groups = json.load(f)

print(f"{len(pairs)} pairs loaded")
for split_type in SPLIT_TYPES:
    print(f"{split_type.upper()}: {len(splits[split_type])} folds")
for g in holdout_groups:
    print(f"fold {g['fold']}: {len(g['cell_lines_held_out'])} cell lines, "
          f"{len(g['drugs_held_out'])} drugs, "
          f"{len(g['tissues_held_out'])} tissues held out")

## Load omics features, drug fingerprints, and drug graphs

Fingerprints feed `SimpleNeuralNetwork`; graphs feed `DrugGNN`. Graph builder
matches `PLAN.md`'s original `smiles_to_graph` design (atom features: atomic
number, degree, formal charge, hybridization, aromaticity, H count).

In [ ]:
rna = pd.read_csv(DATA_DIR / "gene_expression.csv", index_col=0)
protein = pd.read_csv(DATA_DIR / "proteomics.csv", index_col=0)
drug_smiles = pd.read_csv(DATA_DIR / "drug_smiles.csv")

rna = rna[~rna.index.duplicated(keep="first")]
protein = protein[~protein.index.duplicated(keep="first")]

OMICS = {"rna": rna, "protein": protein}


def build_drug_fingerprints(drug_smiles_df: pd.DataFrame, radius: int = 2, n_bits: int = 2048) -> dict:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = {}
    for _, row in drug_smiles_df.iterrows():
        mol = Chem.MolFromSmiles(row["canonical_smiles"])
        if mol is None:
            continue
        fp = generator.GetFingerprint(mol)
        fps[row[COL_DRUG]] = np.array(fp, dtype=np.float32)
    return fps


def atom_to_features(atom) -> list[float]:
    return [
        atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
        int(atom.GetHybridization()), int(atom.GetIsAromatic()), atom.GetTotalNumHs(),
    ]


def smiles_to_graph(smiles: str) -> Data | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = torch.tensor([atom_to_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)


def build_drug_graphs(drug_smiles_df: pd.DataFrame) -> dict:
    graphs = {}
    for _, row in drug_smiles_df.iterrows():
        g = smiles_to_graph(row["canonical_smiles"])
        if g is not None:
            graphs[row[COL_DRUG]] = g
    return graphs


drug_fp = build_drug_fingerprints(drug_smiles)
drug_graphs = build_drug_graphs(drug_smiles)
print(f"Fingerprints: {len(drug_fp)} / {drug_smiles[COL_DRUG].nunique()} drugs")
print(f"Graphs:       {len(drug_graphs)} / {drug_smiles[COL_DRUG].nunique()} drugs")

## Feature builders

`build_feature_matrix` (omics + fingerprint, concatenated) feeds `SimpleNeuralNetwork`
— same shape as notebook 07's sklearn baselines. `build_omics_matrix` (omics only,
drug kept separate as a graph reference) feeds `DrugGNN`.

In [ ]:
def build_feature_matrix(idx: np.ndarray, arm: str) -> tuple[np.ndarray, np.ndarray]:
    sub = pairs.loc[idx]
    omics_X = OMICS[arm].loc[sub[COL_CELLOSAURUS]].to_numpy()
    drug_X = np.vstack([drug_fp[d] for d in sub[COL_DRUG]])
    X = np.hstack([omics_X, drug_X]).astype(np.float32)
    y = sub[COL_IC50].to_numpy().astype(np.float32)
    return X, y


def build_omics_matrix(idx: np.ndarray, arm: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    sub = pairs.loc[idx]
    X = OMICS[arm].loc[sub[COL_CELLOSAURUS]].to_numpy().astype(np.float32)
    y = sub[COL_IC50].to_numpy().astype(np.float32)
    drug_ids = sub[COL_DRUG].to_numpy()
    return X, y, drug_ids


def select_top_variance_columns(X_ref: np.ndarray, k: int, n_extra_cols: int = 0) -> np.ndarray:
    """Top-k highest-variance omics columns, computed from X_ref only. n_extra_cols
    accounts for a concatenated drug fingerprint that's always kept (not variance-selected)."""
    n_omics = X_ref.shape[1] - n_extra_cols
    variances = X_ref[:, :n_omics].var(axis=0)
    top_idx = np.argsort(variances)[-k:]
    return np.concatenate([top_idx, np.arange(n_omics, X_ref.shape[1])])

## Group-aware validation split

Identical logic to notebook 07 — mirrors drevalpy's `_leave_group_out_cv` /
`_leave_pair_out_cv` exactly.

In [ ]:
def make_validation_indices(train_idx: np.ndarray, split_type: str) -> tuple[np.ndarray, np.ndarray]:
    if split_type == "lpo":
        train_inner, val_idx = train_test_split(
            train_idx, test_size=VALIDATION_RATIO, shuffle=True, random_state=RANDOM_STATE
        )
        return np.array(train_inner), np.array(val_idx)

    group_col = {"lco": COL_CELLOSAURUS, "ldo": COL_DRUG, "lto": COL_TISSUE}[split_type]
    train_groups_all = pairs.loc[train_idx, group_col]
    unique_groups = train_groups_all.unique()
    train_groups, val_groups = train_test_split(
        unique_groups, test_size=VALIDATION_RATIO, shuffle=True, random_state=RANDOM_STATE
    )
    train_inner = train_idx[train_groups_all.isin(train_groups).to_numpy()]
    val_idx = train_idx[train_groups_all.isin(val_groups).to_numpy()]
    return train_inner, val_idx

## `SimpleNeuralNetwork` architecture and training

Architecture matches `FeedForwardNetwork` exactly: each hidden layer is
Linear → BatchNorm → Dropout → ReLU, **except the last hidden layer**, which is
just Linear → ReLU (no BN/dropout) before the final unactivated output layer —
that asymmetry is in the original code, not a simplification here. `dropout_prob=0.3`
and `max_epochs=100` are fixed (drevalpy's yaml lists one value for each, so they're
not actually tuned); `units_per_layer` is the real grid dimension (4 architectures);
`batch_size=32`, `patience=5`, Adam with default `lr=1e-3` all match upstream.

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, input_dim: int, units_per_layer: list[int], dropout_prob: float = 0.3):
        super().__init__()
        self.fc_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()

        self.fc_layers.append(nn.Linear(input_dim, units_per_layer[0]))
        self.bn_layers.append(nn.BatchNorm1d(units_per_layer[0]))
        for i in range(1, len(units_per_layer)):
            self.fc_layers.append(nn.Linear(units_per_layer[i - 1], units_per_layer[i]))
            self.bn_layers.append(nn.BatchNorm1d(units_per_layer[i]))
        self.fc_layers.append(nn.Linear(units_per_layer[-1], 1))
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        for i in range(len(self.fc_layers) - 2):
            x = self.fc_layers[i](x)
            x = self.bn_layers[i](x)
            x = self.dropout(x)
            x = F.relu(x)
        x = F.relu(self.fc_layers[-2](x))
        x = self.fc_layers[-1](x)
        return x.squeeze(-1)


def train_simple_nn(X_train, y_train, X_val, y_val, units_per_layer,
                     dropout_prob=0.3, max_epochs=100, batch_size=32, patience=5, lr=1e-3):
    model = FeedForwardNetwork(X_train.shape[1], units_per_layer, dropout_prob).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    X_val_t = torch.from_numpy(X_val).to(DEVICE)
    y_val_t = torch.from_numpy(y_val).to(DEVICE)

    best_val_loss, best_state, epochs_no_improve = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss


SNN_GRID = {"units_per_layer": [[32, 16, 8, 4], [128, 64, 32], [64, 64, 32], [1024, 128, 64, 16]]}


def fit_predict_simple_nn(split_type: str, fold: dict, arm: str) -> dict:
    train_idx, test_idx = np.array(fold["train"]), np.array(fold["test"])
    train_inner_idx, val_idx = make_validation_indices(train_idx, split_type)

    X_train_inner, y_train_inner = build_feature_matrix(train_inner_idx, arm)
    X_val, y_val = build_feature_matrix(val_idx, arm)
    X_test, y_test = build_feature_matrix(test_idx, arm)

    feat_idx = select_top_variance_columns(X_train_inner, TOP_K_FEATURES, n_extra_cols=2048)
    scaler = StandardScaler().fit(X_train_inner[:, feat_idx])
    X_train_inner_s = scaler.transform(X_train_inner[:, feat_idx]).astype(np.float32)
    X_val_s = scaler.transform(X_val[:, feat_idx]).astype(np.float32)
    X_test_s = scaler.transform(X_test[:, feat_idx]).astype(np.float32)

    best_val_loss, best_model, best_params = float("inf"), None, None
    for params in ParameterGrid(SNN_GRID):
        model, val_loss = train_simple_nn(X_train_inner_s, y_train_inner, X_val_s, y_val, **params)
        if val_loss < best_val_loss:
            best_val_loss, best_model, best_params = val_loss, model, params

    best_model.eval()
    with torch.no_grad():
        pred_test = best_model(torch.from_numpy(X_test_s).to(DEVICE)).cpu().numpy()

    r, _ = pearsonr(pred_test, y_test)
    rmse = root_mean_squared_error(y_test, pred_test)
    return {
        "n_train": len(train_inner_idx), "n_test": len(test_idx), "best_params": best_params,
        "pearson_r": r, "rmse": rmse, "test_idx": test_idx, "y_test": y_test, "pred_test": pred_test,
    }

## `DrugGNN` architecture and training

Architecture matches `DrugGraphNet` exactly: 3-layer `GCNConv` drug encoder
(`hidden_dim → hidden_dim*2 → hidden_dim*4`) with `global_mean_pool`, a 2-layer
MLP cell-line encoder, concatenated and passed through a combiner head to a
scalar output. `hidden_dim` and `dropout` are the real grid (4 combinations);
`learning_rate=0.001` and `epochs=100` are fixed (single value in drevalpy's
yaml); `batch_size=1024`, `patience=5` match upstream defaults.

In [ ]:
class DrugGraphNet(nn.Module):
    def __init__(self, num_node_features, num_cell_features, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        from torch_geometric.nn import GCNConv, global_mean_pool
        self.global_mean_pool = global_mean_pool

        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim * 2)
        self.conv3 = GCNConv(hidden_dim * 2, hidden_dim * 4)
        self.drug_embed_fc = nn.Linear(hidden_dim * 4, hidden_dim)

        self.cell_fc1 = nn.Linear(num_cell_features, hidden_dim * 2)
        self.cell_fc2 = nn.Linear(hidden_dim * 2, hidden_dim)

        self.combiner_fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.combiner_fc2 = nn.Linear(hidden_dim, 32)
        self.output_fc = nn.Linear(32, 1)

    def forward(self, drug_graph, cell_features):
        x, edge_index, batch = drug_graph.x, drug_graph.edge_index, drug_graph.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))

        drug_embedding = self.drug_embed_fc(self.global_mean_pool(x, batch))

        cell_embedding = F.relu(self.cell_fc1(cell_features))
        cell_embedding = F.dropout(cell_embedding, p=self.dropout, training=self.training)
        cell_embedding = self.cell_fc2(cell_embedding)

        combined = torch.cat([drug_embedding, cell_embedding], dim=1)
        x = F.relu(self.combiner_fc1(combined))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.combiner_fc2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.output_fc(x).view(-1)


class DrugCellDataset(Dataset):
    def __init__(self, X_omics, y, drug_ids, graphs):
        self.X_omics = torch.from_numpy(X_omics)
        self.y = torch.from_numpy(y)
        self.drug_ids = drug_ids
        self.graphs = graphs

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.graphs[self.drug_ids[i]], self.X_omics[i], self.y[i]


def collate_drug_cell(batch):
    graphs, cell_feats, ys = zip(*batch)
    return Batch.from_data_list(list(graphs)), torch.stack(cell_feats), torch.stack(ys)


def train_drug_gnn(X_train, y_train, drug_ids_train, X_val, y_val, drug_ids_val,
                    hidden_dim=64, dropout=0.2, lr=0.001, max_epochs=100, batch_size=1024, patience=5):
    num_node_features = next(iter(drug_graphs.values())).x.shape[1]
    model = DrugGraphNet(num_node_features, X_train.shape[1], hidden_dim, dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_loader = DataLoader(
        DrugCellDataset(X_train, y_train, drug_ids_train, drug_graphs),
        batch_size=batch_size, shuffle=True, collate_fn=collate_drug_cell,
    )
    val_loader = DataLoader(
        DrugCellDataset(X_val, y_val, drug_ids_val, drug_graphs),
        batch_size=batch_size, shuffle=False, collate_fn=collate_drug_cell,
    )

    best_val_loss, best_state, epochs_no_improve = float("inf"), None, 0
    for epoch in range(max_epochs):
        model.train()
        for batch_graph, cell_feats, ys in train_loader:
            batch_graph, cell_feats, ys = batch_graph.to(DEVICE), cell_feats.to(DEVICE), ys.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_graph, cell_feats), ys)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss_sum, n_val = 0.0, 0
        with torch.no_grad():
            for batch_graph, cell_feats, ys in val_loader:
                batch_graph, cell_feats, ys = batch_graph.to(DEVICE), cell_feats.to(DEVICE), ys.to(DEVICE)
                pred = model(batch_graph, cell_feats)
                val_loss_sum += loss_fn(pred, ys).item() * len(ys)
                n_val += len(ys)
        val_loss = val_loss_sum / n_val

        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss


DGNN_GRID = {"hidden_dim": [64, 128], "dropout": [0.2, 0.3]}


def fit_predict_drug_gnn(split_type: str, fold: dict, arm: str) -> dict:
    train_idx, test_idx = np.array(fold["train"]), np.array(fold["test"])
    train_inner_idx, val_idx = make_validation_indices(train_idx, split_type)

    X_train_inner, y_train_inner, drug_train_inner = build_omics_matrix(train_inner_idx, arm)
    X_val, y_val, drug_val = build_omics_matrix(val_idx, arm)
    X_test, y_test, drug_test = build_omics_matrix(test_idx, arm)

    feat_idx = select_top_variance_columns(X_train_inner, TOP_K_FEATURES, n_extra_cols=0)
    scaler = StandardScaler().fit(X_train_inner[:, feat_idx])
    X_train_inner_s = scaler.transform(X_train_inner[:, feat_idx]).astype(np.float32)
    X_val_s = scaler.transform(X_val[:, feat_idx]).astype(np.float32)
    X_test_s = scaler.transform(X_test[:, feat_idx]).astype(np.float32)

    best_val_loss, best_model, best_params = float("inf"), None, None
    for params in ParameterGrid(DGNN_GRID):
        model, val_loss = train_drug_gnn(
            X_train_inner_s, y_train_inner, drug_train_inner, X_val_s, y_val, drug_val, **params
        )
        if val_loss < best_val_loss:
            best_val_loss, best_model, best_params = val_loss, model, params

    best_model.eval()
    test_loader = DataLoader(
        DrugCellDataset(X_test_s, y_test, drug_test, drug_graphs),
        batch_size=1024, shuffle=False, collate_fn=collate_drug_cell,
    )
    preds = []
    with torch.no_grad():
        for batch_graph, cell_feats, _ in test_loader:
            batch_graph, cell_feats = batch_graph.to(DEVICE), cell_feats.to(DEVICE)
            preds.append(best_model(batch_graph, cell_feats).cpu().numpy())
    pred_test = np.concatenate(preds)

    r, _ = pearsonr(pred_test, y_test)
    rmse = root_mean_squared_error(y_test, pred_test)
    return {
        "n_train": len(train_inner_idx), "n_test": len(test_idx), "best_params": best_params,
        "pearson_r": r, "rmse": rmse, "test_idx": test_idx, "y_test": y_test, "pred_test": pred_test,
    }

## Run setup

Shared across the two model-specific cells below — same pattern as notebook 07,
each model runnable/re-runnable independently.

In [ ]:
run_splits = {"lco": [splits["lco"][0]]} if QUICK_TEST else splits
results_rows = []
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def save_predictions(split_type: str, arm: str, model_name: str, fold_i: int, out: dict) -> None:
    pred_dir = RESULTS_DIR / split_type / arm / model_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "row_idx": out["test_idx"], "response": out["y_test"], "predictions": out["pred_test"],
    }).to_csv(pred_dir / f"predictions_fold_{fold_i}.csv", index=False)


def log_result(split_type: str, fold_i: int, arm: str, model_name: str, out: dict,
                elapsed: float, best_params: dict | None = None) -> None:
    results_rows.append({
        "split": split_type.upper(), "fold": fold_i, "arm": arm, "model": model_name,
        "n_train": out["n_train"], "n_test": out["n_test"],
        "pearson_r": round(out["pearson_r"], 4), "rmse": round(out["rmse"], 4),
        "best_params": best_params, "seconds": round(elapsed, 1),
    })
    print(f"{split_type.upper()} fold {fold_i} | {arm:8s} | {model_name:18s} "
          f"| r={out['pearson_r']:.3f} rmse={out['rmse']:.3f} | best={best_params} | {elapsed:.0f}s")

### Run — `SimpleNeuralNetwork` (RNA-only, then protein-only)

In [ ]:
for split_type, folds in run_splits.items():
    for fold_i, fold in enumerate(folds):
        for arm in ARMS:
            start = time.time()
            out = fit_predict_simple_nn(split_type, fold, arm)
            elapsed = time.time() - start
            save_predictions(split_type, arm, "SimpleNeuralNetwork", fold_i, out)
            log_result(split_type, fold_i, arm, "SimpleNeuralNetwork", out, elapsed, out["best_params"])

### Run — `DrugGNN` (RNA-only, then protein-only)

In [ ]:
for split_type, folds in run_splits.items():
    for fold_i, fold in enumerate(folds):
        for arm in ARMS:
            start = time.time()
            out = fit_predict_drug_gnn(split_type, fold, arm)
            elapsed = time.time() - start
            save_predictions(split_type, arm, "DrugGNN", fold_i, out)
            log_result(split_type, fold_i, arm, "DrugGNN", out, elapsed, out["best_params"])

## Save + quick summary

Dedupes on (split, fold, arm, model), keeping the most recent row — safe to
re-run either model cell above independently.

In [ ]:
summary = pd.DataFrame(results_rows).drop_duplicates(
    subset=["split", "fold", "arm", "model"], keep="last"
)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary.groupby(["split", "arm", "model"])[["pearson_r", "rmse"]].mean()